In [7]:
import os
import json

def list_base_filenames(paths, suffix, remove_trailing_underscore=False):
    base_files = set()
    for path in paths:
        for file_name in os.listdir(path):
            if file_name.endswith(suffix):
                base_name = file_name.replace(suffix, '')
                if remove_trailing_underscore and base_name.endswith('_'):
                    base_name = base_name[:-1] 
                base_files.add(base_name)
    return base_files

def select_best_candidate(paths, file_name, suffix):
    best_data = None
    best_accuracy = -1

    for path in paths:
        full_name = file_name + suffix
        file_path = os.path.join(path, full_name)
        #print(f"Checking: {file_path}")  # <-- Aqui!
        if not os.path.exists(file_path):
            continue
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            accuracy = data.get('summary', {}).get('accuracy', 0)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_data = data

    return best_data

def analyze_corrections(simple_paths, agentic_paths):
    # Alinha pelos nomes base
    simple_bases = list_base_filenames(simple_paths, '_output_simple.json')
    # print("simple")
    # print(simple_bases)
    agentic_bases = list_base_filenames(agentic_paths, '_output.json', remove_trailing_underscore=True)
    # print("agents")
    # print(agentic_bases)
    common_bases = simple_bases & agentic_bases
    
    # print("comon")
    # print(common_bases)

    for base_name in sorted(common_bases):
        # Seleciona os melhores candidatos
        simple_data = select_best_candidate(simple_paths, base_name, '_output_simple.json')
        agentic_data = select_best_candidate(agentic_paths, base_name, '_output.json')

        if not simple_data or not agentic_data:
            print(f"Skipping {base_name}, missing data.")
            continue

        simple_results = simple_data.get('results', [])
        agentic_results = agentic_data.get('results', [])

        total_entries = 0
        false_to_true_with_tool = 0
        true_to_false = 0

        for simple_item, agentic_item in zip(simple_results, agentic_results):
            total_entries += 1
            before_correct = simple_item.get('isCorrect')
            after_correct = agentic_item.get('isCorrect')
            tool_call_str = agentic_item.get('tool_call', '')

            if before_correct is False and after_correct is True and tool_call_str:
                false_to_true_with_tool += 1
            if before_correct is True and after_correct is False:
                true_to_false += 1

        percent_false_to_true = (false_to_true_with_tool / total_entries * 100) if total_entries else 0
        percent_true_to_false = (true_to_false / total_entries * 100) if total_entries else 0

        # Exibe resultados por arquivo base
        print(f"\nFile Base: {base_name}")
        print(f"  Total entries: {total_entries}")
        print(f"  False -> True with tool_call: {false_to_true_with_tool} ({percent_false_to_true:.2f}%)")
        print(f"  True -> False: {true_to_false} ({percent_true_to_false:.2f}%)")


In [8]:
simple_paths = ['simple\\processed\\run1_corrected',
                'simple\\processed\\run2_corrected',
                'simple\\processed\\run3_corrected']
agentic_paths = ['agentic\\processed\\run1_corrected', 
                 'agentic\\processed\\run2_corrected',
                 'agentic\\processed\\run3_corrected']
analyze_corrections(simple_paths, agentic_paths)


File Base: command-r7b
  Total entries: 38
  False -> True with tool_call: 4 (10.53%)
  True -> False: 2 (5.26%)

File Base: deepseek-r1_14b
  Total entries: 38
  False -> True with tool_call: 3 (7.89%)
  True -> False: 5 (13.16%)

File Base: deepseek-r1_32b
  Total entries: 38
  False -> True with tool_call: 1 (2.63%)
  True -> False: 2 (5.26%)

File Base: gemma3_12b
  Total entries: 38
  False -> True with tool_call: 3 (7.89%)
  True -> False: 5 (13.16%)

File Base: gemma3_27b
  Total entries: 38
  False -> True with tool_call: 1 (2.63%)
  True -> False: 5 (13.16%)

File Base: qwen2.5_14b
  Total entries: 38
  False -> True with tool_call: 2 (5.26%)
  True -> False: 7 (18.42%)

File Base: qwen2.5_32b
  Total entries: 38
  False -> True with tool_call: 4 (10.53%)
  True -> False: 5 (13.16%)

File Base: qwq
  Total entries: 38
  False -> True with tool_call: 5 (13.16%)
  True -> False: 1 (2.63%)
